# Introduction to Deep Learning, Assignment 2, Task 2


# Function definitions for creating the datasets

First we need to create our datasets that are going to be used for training our models.

In order to create image queries of simple arithmetic operations such as '15+13' or '42-10' we need to create images of '+' and '-' signs using ***open-cv*** library. We will use these operand signs together with the MNIST dataset to represent the digits.

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import Dense, RNN, LSTM, Flatten, TimeDistributed, LSTMCell
from tensorflow.keras.layers import RepeatVector, Conv2D, SimpleRNN, GRU, Reshape, ConvLSTM2D, Conv2DTranspose

2025-12-14 13:32:51.040153: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-14 13:32:51.068720: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-14 13:32:56.759316: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from scipy.ndimage import rotate


# Create plus/minus operand signs
def generate_images(number_of_images=50, sign='-'):
    blank_images = np.zeros([number_of_images, 28, 28])  # Dimensionality matches the size of MNIST images (28x28)
    x = np.random.randint(12, 16, (number_of_images, 2)) # Randomized x coordinates
    y1 = np.random.randint(6, 10, number_of_images)       # Randomized y coordinates
    y2 = np.random.randint(18, 22, number_of_images)     # -||-

    for i in range(number_of_images): # Generate n different images
        cv2.line(blank_images[i], (y1[i], x[i,0]), (y2[i], x[i, 1]), (255,0,0), 2, cv2.LINE_AA)     # Draw lines with randomized coordinates
        if sign == '+':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA) # Draw lines with randomized coordinates

    return blank_images

def show_generated(images, n=5):
    plt.figure(figsize=(2, 2))
    for i in range(n**2):
        plt.subplot(n, n, i+1)
        plt.axis('off')
        plt.imshow(images[i])
    plt.show()

In [3]:
def create_data(highest_integer, num_addends=2, operands=['+', '-']):
    """
    Creates the following data for all pairs of integers up to [1:highest integer][+/-][1:highest_integer]:

    @return:
    X_text: '51+21' -> text query of an arithmetic operation (5)
    X_img : Stack of MNIST images corresponding to the query (5 x 28 x 28) -> sequence of 5 images of size 28x28
    y_text: '72' -> answer of the arithmetic text query
    y_img :  Stack of MNIST images corresponding to the answer (3 x 28 x 28)

    Images for digits are picked randomly from the whole MNIST dataset.
    """

    num_indices = [np.where(MNIST_labels==x) for x in range(10)]
    num_data = [MNIST_data[inds] for inds in num_indices]
    image_mapping = dict(zip(unique_characters[:10], num_data))
    image_mapping['-'] = generate_images()
    image_mapping['+'] = generate_images(sign='+')
    image_mapping['*'] = generate_images(sign='*')
    image_mapping[' '] = np.zeros([1, 28, 28])

    X_text, X_img, y_text, y_img = [], [], [], []

    for i in range(highest_integer + 1):      # First addend
        for j in range(highest_integer + 1):  # Second addend
            for sign in operands: # Create all possible combinations of operands
                query_string = to_padded_chars(str(i) + sign + str(j), max_len=max_query_length, pad_right=True)
                query_image = []
                for n, char in enumerate(query_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    query_image.append(image_set[index].squeeze())

                result = eval(query_string)
                result_string = to_padded_chars(result, max_len=max_answer_length, pad_right=True)
                result_image = []
                for n, char in enumerate(result_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    result_image.append(image_set[index].squeeze())

                X_text.append(query_string)
                X_img.append(np.stack(query_image))
                y_text.append(result_string)
                y_img.append(np.stack(result_image))

    return np.stack(X_text), np.stack(X_img)/255., np.stack(y_text), np.stack(y_img)/255.

def to_padded_chars(integer, max_len=3, pad_right=False):
    """
    Returns a string of len()=max_len, containing the integer padded with ' ' on either right or left side
    """
    length = len(str(integer))
    padding = (max_len - length) * ' '
    if pad_right:
        return str(integer) + padding
    else:
        return padding + str(integer)


# Creating our data

The dataset consists of 20000 samples that (additions and subtractions between all 2-digit integers) and they have two kinds of inputs and label modalities:

  **X_text**: strings containing queries of length 5: ['  1+1  ', '11-18', ...]

  **X_image**: a stack of images representing a single query, dimensions: [5, 28, 28]

  **y_text**: strings containing answers of length 3: ['  2', '156']

  **y_image**: a stack of images that represents the answer to a query, dimensions: [3, 28, 28]

In [4]:
# Illustrate the generated query/answer pairs

unique_characters = '0123456789+- '       # All unique characters that are used in the queries (13 in total: digits 0-9, 2 operands [+, -], and a space character ' '.)
highest_integer = 99                      # Highest value of integers contained in the queries

max_int_length = len(str(highest_integer))# Maximum number of characters in an integer
max_query_length = max_int_length * 2 + 1 # Maximum length of the query string (consists of two integers and an operand [e.g. '22+10'])
max_answer_length = 3    # Maximum length of the answer string (the longest resulting query string is ' 1-99'='-98')

# Create the data (might take around a minute)
(MNIST_data, MNIST_labels), _ = tf.keras.datasets.mnist.load_data()
X_text, X_img, y_text, y_img = create_data(highest_integer)
print(X_text.shape, X_img.shape, y_text.shape, y_img.shape)




(20000,) (20000, 5, 28, 28) (20000,) (20000, 3, 28, 28)


# My own helper functions

In the models below teacher forcing is used. For this the vocabulary will need a start and end token. Subsequently the one-hot encoding and decoding functions need to be altered to include these.

In [5]:
# Teacher forcing preparation
vocabulary_tf = list(unique_characters)+['<start>','<end>'] 

indices = {char:i for i, char in enumerate(vocabulary_tf)}
reverse_indices={i:char for i,char in enumerate(vocabulary_tf)}


# One-hot encoding
def encode_labels_tf(labels, vocabulary=vocabulary_tf, indices_map=indices):
  n = len(labels)
  length = len(labels[0])+2 # for start of sequence <start> and end of sequence <end> tokens.

  one_hot = np.zeros([n, length, len(vocabulary)])
  for i, label in enumerate(labels):
    full_label = ['<start>'] + list(label) + ['<end>']
    m = np.zeros([length, len(vocabulary)])
    for j, char in enumerate(full_label):
       m[j, indices_map[char]] = 1
    one_hot[i] = m

  return one_hot

# One-hot decoding
def decode_labels_tf(labels, vocabulary=vocabulary_tf, indices_map = reverse_indices):
    pred = np.argmax(labels, axis=2)
    predicted = [''.join([indices_map[i] for i in j]) for j in pred]

    return predicted

X_text_onehot = encode_labels_tf(X_text)
y_text_onehot = encode_labels_tf(y_text)

print(X_text_onehot.shape, y_text_onehot.shape)

(20000, 7, 15) (20000, 5, 15)


# Model with attention

In [6]:
size=0.1

X_train, X_test, y_train, y_test = train_test_split(
    X_img, y_text_onehot, random_state=42, test_size=size
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, random_state=42, test_size=size/(1-size)
)

y_train_in = y_train[:, :-1, :]
y_train_target = y_train[:, 1:, :]

y_val_in = y_val[:, :-1, :]
y_val_target = y_val[:, 1:, :]

y_test_in = y_test[:, :-1, :]
y_test_target = y_test[:, 1:, :]

In [7]:
# Your code is: damn code

from tensorflow.keras.layers import BatchNormalization, Activation, MaxPooling2D, LSTM, TimeDistributed, Dropout, Input, Add, LayerNormalization, Attention, Concatenate
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import L2, L1L2


# First we create a build-encoder function
def build_image2text_encoder(dropout, RLstrength, max_size=512):
    
    # Initialize an encoder
    X_in = Input(shape = (5,28,28,1)) # 5 times a grayscale image


    # Build encoder layers
    ## Block 1
    B1 = TimeDistributed(Conv2D(32, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(X_in)
    B1 = TimeDistributed(BatchNormalization())(B1)
    B1 = TimeDistributed(Activation('relu'))(B1)
    B1 = TimeDistributed(Dropout(dropout))(B1)
    B1_final = TimeDistributed(MaxPooling2D())(B1)


    ## Initialize the residual connection
    residual = TimeDistributed(Activation('linear', name = "residual_branch"))(B1_final)
    residual2= TimeDistributed(Conv2D(64, (1,1), kernel_regularizer=L2(RLstrength/8)))(residual)
    
    ## Block 2
    B2 = TimeDistributed(Conv2D(64, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(B1_final)
    B2 = TimeDistributed(BatchNormalization())(B2)
    B2 = TimeDistributed(Activation('relu'))(B2)
    B2_final = TimeDistributed(Dropout(dropout))(B2)

    ## Connect residual connection to output of two Conv2D blocks
    combined = Add()([B2_final, residual2])
    combined = TimeDistributed(Activation('relu'))(combined)

    ## Final pooling before the ConvLSTM2D layer
    final_pooling = TimeDistributed(MaxPooling2D(name='final_pooling'))(combined)


    ## Recurrent convolutional layers
    output, hidden, cell = ConvLSTM2D(
            filters=128, 
            kernel_size=(3,3), 
            padding='same',
            return_sequences=True, 
            use_bias=True, 
            return_state=True, 
            name='ConvLSTM', 
            dropout=dropout, #dropout
            recurrent_dropout=dropout, #dropout
            kernel_regularizer = L2(RLstrength),
            recurrent_regularizer = L2(RLstrength))(final_pooling) #L2(RLstrength)

    encoder = tf.keras.Model(inputs=X_in, outputs=[output, hidden, cell], name = "encoder_model")
    return encoder





# Here we build the full model
def build_image2text(dropout = 0.3, max_size=512, learning_rate = 2.5e-4, max_answer_length_tf=4,RLstrength=1.0e-4):


    # Define input layer of full model
    X_in = Input(shape = (5,28,28,1), name = 'sequence')
    Y_in = Input(shape=(max_answer_length_tf, len(vocabulary_tf)))


    # Encoder step
    encoder = build_image2text_encoder(dropout,RLstrength)
    output, hidden, cell = encoder(X_in)


    # Bridge step
    ## We take the output hidden and cell state of the encoder so that we may use teacher forcing.
    h_flattened = Flatten(name='h_flattened')(hidden)
    h_initial = Dense(max_size, activation = 'relu',kernel_regularizer=L2(RLstrength), name='h0')(h_flattened)

    c_flattened = Flatten(name='c_flattened')(cell)
    c_initial = Dense(max_size, activation = 'relu',kernel_regularizer=L2(RLstrength),name='c0')(c_flattened)

    ini_state = [h_initial, c_initial]

    # Decoder step
    ## We initialize the decoder using teacher forcing

    lstm = LSTM(
        max_size, 
        return_sequences = True, 
        return_state=True, 
        dropout = dropout, 
        recurrent_dropout = dropout, 
        name='decoder_lstm',
        kernel_regularizer = L2(RLstrength/4),
        recurrent_regularizer = L2(RLstrength/4)
        )(Y_in, initial_state = ini_state) #
    
    dense = TimeDistributed(Dense(len(vocabulary_tf), activation='softmax', kernel_regularizer=L2(RLstrength/4)), name = 'decoder_dense')

    query, _, _ = lstm

    key_flatten = TimeDistributed(Flatten(), name='key_flatten')(output)
    key = TimeDistributed(Dense(max_size, kernel_regularizer = L2(RLstrength/2)), name = 'key_decoder')(key_flatten)
    attention_block = Attention(name='attention_block')([query, key])

    combined = Concatenate(axis=-1, name = 'concat_q_A')([query, attention_block])
    combined = Dense(max_size, name = 'combined_dense', kernel_regularizer = L1L2(l1 = 5.0e-5, l2=RLstrength/2))(combined)

    residual = TimeDistributed(Dense(
        max_size, 
        use_bias=False, 
        kernel_regularizer = L2(RLstrength/2)
        ), 
        name = 'decoder_res_dense'
        )(Y_in)

    final = Add(name= 'decoder_with_residual')([combined, residual])
    final = LayerNormalization(axis=-1, name='decoder_layer_norm')(final)
    y_out = dense(final)


    # Full model step
    full = tf.keras.Model(inputs=[X_in, Y_in], outputs = y_out)
    full.compile(
        loss='categorical_crossentropy', optimizer=Adam(learning_rate=learning_rate), metrics=['categorical_accuracy']
    )

    full.summary(expand_nested=True)
    return full

##### Model initialization and compile

#### Shock system with high LR

In [106]:
LR2 = 1.0e-3

current_optimizer = image2text.optimizer

current_optimizer.learning_rate.assign(LR2)
#image2text.compile(
#        loss='categorical_crossentropy', optimizer=Adam(learning_rate=learning_rate), metrics=['categorical_accuracy']
#    )
early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = "val_loss", #'val_categorical_accuracy', mode = 'max',
    patience=18,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor= "val_loss",#'val_categorical_accuracy', mode = 'max',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

history2 = image2text.fit(x=[X_train, y_train_in], y=y_train_target,
               epochs = 120,
               batch_size = 64,
               validation_data = ([X_val, y_val_in], y_val_target),
               callbacks=[lr_scheduler, early_stopper])

Epoch 1/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - categorical_accuracy: 0.5547 - loss: 2.1721 - val_categorical_accuracy: 0.5895 - val_loss: 1.5672 - learning_rate: 0.0010
Epoch 2/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - categorical_accuracy: 0.6754 - loss: 1.2107 - val_categorical_accuracy: 0.6584 - val_loss: 1.2731 - learning_rate: 0.0010
Epoch 3/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 67ms/step - categorical_accuracy: 0.7365 - loss: 0.9817 - val_categorical_accuracy: 0.6593 - val_loss: 1.2843 - learning_rate: 0.0010
Epoch 4/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 61ms/step - categorical_accuracy: 0.7561 - loss: 0.9216 - val_categorical_accuracy: 0.6944 - val_loss: 1.1661 - learning_rate: 0.0010
Epoch 5/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 62ms/step - categorical_accuracy: 0.7690 - loss: 0.8846 - val_categorical_accuracy: 0.6854 - val_loss: 1.2175 - learning_rate: 0.0010
Epoch 6/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - categorical_accuracy: 0.7794 - loss: 0.8644 - va

#### Inference loop

In [8]:
def Encoder_inference(trained_model_full):
        
    X_in = tf.keras.Input(shape=(5, 28, 28, 1))

    encoder_outputs = trained_model_full.get_layer("encoder_model")
    Sequence, hidden, cell= encoder_outputs(X_in)

    h_initial = trained_model_full.get_layer('h0')(tf.keras.layers.Flatten()(hidden))
    c_initial = trained_model_full.get_layer('c0')(tf.keras.layers.Flatten()(cell))

    keys_flat = TimeDistributed(Flatten())(Sequence)
    keys_decoder = trained_model_full.get_layer('key_decoder')
    keys = keys_decoder(keys_flat)


    encoder_inference = tf.keras.Model(inputs=X_in, outputs=[keys, h_initial, c_initial], name="encoder_inference")
    return encoder_inference


def Decoder_inference(trained_full_model, max_size, vocab_size=len(vocabulary_tf)):
    
    # Input objects
    input_token = Input(shape=(1, vocab_size), name='token_input')
    keys = Input(shape=(None, max_size), name='keys')
    input_h = Input(shape=(max_size,), name='input_h')
    input_c = Input(shape=(max_size,), name='input_c')

    # Here we pull all the trained layers from the model
    lstm = trained_full_model.get_layer('decoder_lstm')
    attention = trained_full_model.get_layer('attention_block')
    lstm.return_sequences = True # for some reason this is necessary
    lstm.return_state = True
    concat = trained_full_model.get_layer('concat_q_A')
    combined_dense = trained_full_model.get_layer('combined_dense')
    res_dense = trained_full_model.get_layer('decoder_res_dense')
    add_res = trained_full_model.get_layer('decoder_with_residual')
    layernorm = trained_full_model.get_layer('decoder_layer_norm')
    final_dense = trained_full_model.get_layer('decoder_dense')
    
    # 
    Query, h, c = lstm(
        input_token, 
        initial_state=[input_h, input_c]
    )

    attention_values = attention([Query, keys])
    combined_attention_output = concat([Query, attention_values])
    mixed_features = combined_dense(combined_attention_output)

    # apply residual connection
    residual = res_dense(input_token)
    residual_added = add_res([mixed_features, residual])

    layernorm_final = layernorm(residual_added)
    prediction = final_dense(layernorm_final)

    decoder_inference_model = tf.keras.Model(
        inputs=[input_token, keys, input_h, input_c],
        outputs=[prediction, h, c],
    )
    
    return decoder_inference_model

In [9]:
from tensorflow.keras.utils import to_categorical

def generate_prediction_sequence(
    X_sample, 
    encoder_inference_model, 
    decoder_inference_model, 
    max_answer_length_tf, 
    indices, 
    reverse_indices, 
    vocab_size,
):
    keys, h_initial, c_initial = encoder_inference_model.predict(X_sample, verbose=0)
    states = [h_initial, c_initial]
    start_token = indices['<start>']
    input_token = to_categorical(start_token, num_classes=vocab_size).reshape(1, 1, vocab_size)
    predicted_indices = []

    for t in range(max_answer_length_tf):
        decoder_input = [input_token, keys] + states
        predictions_output = decoder_inference_model.predict(decoder_input, verbose=0)
        predictions = predictions_output[0]
        h = predictions_output[1]
        c = predictions_output[2]
        predicted_index = np.argmax(predictions[0, 0, :])#[0, -1, :]
        predicted_indices.append(predicted_index)
        if predicted_index == indices['<end>']:
            break
        
        states = [h, c]
        input_token = to_categorical(predicted_index, num_classes=vocab_size).reshape(1, 1, vocab_size)

    string = "".join([
        reverse_indices[p]
        for p in predicted_indices
        if reverse_indices[p] not in ["<start>", "<end>"]
    ])

    return string


In [10]:
def evaluate_test_set(start, N_samples, printing = False):
    counter_math_correctness = 0
    counter_token_correctness = 0
    for sample in range(start, start+N_samples):
        X_sample = X_test[sample]
        X_sample_reshaped = np.expand_dims(X_sample, axis=-1)
        X_sample_reshaped2 = np.expand_dims(X_sample_reshaped, axis=0)
        max_answer_length_tf=4
        vocab_size = len(vocabulary_tf)

        sequence = generate_prediction_sequence(
            X_sample_reshaped2, 
            encoder_inference, 
            decoder_inference, 
            max_answer_length_tf, 
            indices, 
            reverse_indices, 
            vocab_size,    
        )

        ground_truth_index = np.argmax(y_test_target[sample],axis=-1)
        ground_truth_string = "".join([reverse_indices[index] for index in ground_truth_index
                                        if reverse_indices[index] not in ["<start>", "<end>"]])

        if printing== True:
            print(f"Actual string = {ground_truth_string}. Predicted string = {sequence}")

        if ground_truth_string == sequence:
            counter_math_correctness +=1
        
        for token in range(min(len(ground_truth_string), len(sequence))):
            if ground_truth_string[token]==sequence[token]:
                counter_token_correctness+=1
        
        print(f"step {sample-start} out of {N_samples}")
    
    math_accuracy = counter_math_correctness/N_samples
    token_accuracy = counter_token_correctness/(3*N_samples)

    print(f"math accuracy = {math_accuracy}. token accuracy = {token_accuracy}")

    return math_accuracy, token_accuracy





#### Function application

In [ ]:
dropout=0.55
learning_rate=5.0e-5
RLstrength=1.0e-4
max_size=256
stopper_patience = 60
scheduler_patience = 20

image2text = build_image2text(dropout=dropout, learning_rate=learning_rate, RLstrength=RLstrength, max_size=max_size)

history = image2text.fit(x=[X_train, y_train_in], y=y_train_target, 
               epochs = 40,
               batch_size = 64,
               validation_data = ([X_val, y_val_in], y_val_target))

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)


#LR_start = 2.0e-4
#current_optimizer = image2text.optimizer
#current_optimizer.learning_rate.assign(LR_start)

history = image2text.fit(x=[X_train, y_train_in], y=y_train_target, 
               epochs = 120,
               batch_size = 64,
               validation_data = ([X_val, y_val_in], y_val_target),
               callbacks=[lr_scheduler, early_stopper])

#LR2 = 1.0e-5

#current_optimizer = image2text.optimizer

#current_optimizer.learning_rate.assign(LR2)
#image2text.compile(
#        loss='categorical_crossentropy', optimizer=Adam(learning_rate=learning_rate), metrics=['categorical_accuracy']
#    )
"""
history2 = image2text.fit(x=[X_train, y_train_in], y=y_train_target,
               epochs = 120,
               batch_size = 32,
               validation_data = ([X_val, y_val_in], y_val_target),
               callbacks=[lr_scheduler, early_stopper])"""

Model: "functional_19"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_model       │ [(None, 5, 7, 7,  │    906,560 │ sequence[0][0]    │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ input_layer_30 │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        320 │ -                 │
│ time_distributed_1… │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        128 │ -                 │
│ time_distributed_1… │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_1… │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_1… │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_1… │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │     18,496 │ -                 │
│ time_distributed_1… │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │        256 │ -                 │
│ time_distributed_1… │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_1… │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_1… │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_1… │ 64)               │            │                 

 Total params: 6,142,287 (23.43 MB)

 Trainable params: 6,142,095 (23.43 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/40


E0000 00:00:1765733053.493106 1744603 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/functional_19_1/encoder_model_1/ConvLSTM_1/while/body/_108/functional_19_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_8' -> 'StatefulPartitionedCall/functional_19_1/encoder_model_1/ConvLSTM_1/while/body/_108/functional_19_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/functional_19_1/encoder_model_1/ConvLSTM_1/while/body/_108/functional_19_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Sigmoid' -> 'StatefulPartitionedCall/functional_19_1/encoder_model_1/ConvLSTM_1/while/body/_108/functional_19_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_9', 'StatefulPartitionedCall/functional_19_1/encoder_model_1/ConvLSTM_1/while/body/_108/functional_19_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Sigmoid_2' -> 'StatefulPartitionedCall/

250/250 ━━━━━━━━━━━━━━━━━━━━ 22s 71ms/step - categorical_accuracy: 0.4610 - loss: 2.9859 - val_categorical_accuracy: 0.4814 - val_loss: 2.7112
Epoch 2/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - categorical_accuracy: 0.5142 - loss: 2.5873 - val_categorical_accuracy: 0.4834 - val_loss: 2.5223
Epoch 3/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.5326 - loss: 2.3605 - val_categorical_accuracy: 0.4855 - val_loss: 2.4039
Epoch 4/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.5449 - loss: 2.1810 - val_categorical_accuracy: 0.5040 - val_loss: 2.2473
Epoch 5/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.5513 - loss: 2.0608 - val_categorical_accuracy: 0.5246 - val_loss: 2.1147
Epoch 6/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.5528 - loss: 1.9738 - val_categorical_accuracy: 0.5285 - val_loss: 2.0278
Epoch 7/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - categorical_accuracy: 0.5563 - 

'\nhistory2 = image2text.fit(x=[X_train, y_train_in], y=y_train_target,\n               epochs = 120,\n               batch_size = 32,\n               validation_data = ([X_val, y_val_in], y_val_target),\n               callbacks=[lr_scheduler, early_stopper])'

In [28]:
optimal_acc = 0.42 #choose your optimal thingetje

500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - categorical_accuracy: 0.7835 - loss: 0.8334


[0.8334107398986816, 0.7834864854812622]

In [43]:
encoder_inference = Encoder_inference(
    image2text
)

decoder_inference = Decoder_inference(
    image2text, max_size=max_size
)

start = random.randint(0,1000)
N_samples = 200
X_sample_set = X_test[start: start+N_samples]
math_accuracy, token_accuracy = evaluate_test_set(start, N_samples)

loss, acc =image2text.evaluate(
    x=[X_train, y_train_in],
    y=y_train_target
)

val_loss, val_acc = image2text.evaluate(
    x=[X_val, y_val_in],
    y=y_val_target
)

print(f"dropout = {dropout}, RLstrength = {RLstrength} produces: math accuracy = {math_accuracy}, token accuracy = {token_accuracy}")
print(f"acc = {acc}, loss = {loss}")
print(f"val_acc = {val_acc}, val_loss = {val_loss}")

if math_accuracy > optimal_acc:
    optimal_acc = math_accuracy
    image2text.save(f'math_acc_{math_accuracy:3f}.keras')

step 0 out of 200
step 1 out of 200
step 2 out of 200
step 3 out of 200
step 4 out of 200
step 5 out of 200
step 6 out of 200
step 7 out of 200
step 8 out of 200
step 9 out of 200
step 10 out of 200
step 11 out of 200
step 12 out of 200
step 13 out of 200
step 14 out of 200
step 15 out of 200
step 16 out of 200
step 17 out of 200
step 18 out of 200
step 19 out of 200
step 20 out of 200
step 21 out of 200
step 22 out of 200
step 23 out of 200
step 24 out of 200
step 25 out of 200
step 26 out of 200
step 27 out of 200
step 28 out of 200
step 29 out of 200
step 30 out of 200
step 31 out of 200
step 32 out of 200
step 33 out of 200
step 34 out of 200
step 35 out of 200
step 36 out of 200
step 37 out of 200
step 38 out of 200
step 39 out of 200
step 40 out of 200
step 41 out of 200
step 42 out of 200
step 43 out of 200
step 44 out of 200
step 45 out of 200
step 46 out of 200
step 47 out of 200
step 48 out of 200
step 49 out of 200
step 50 out of 200
step 51 out of 200
step 52 out of 200
ste

dropout = 0.55, RLstrength = 0.001 produces: math accuracy = 0.42. token accuracy = 0.7266666666666667

dropout = 0.6, RLstrength = 0.001 producess: math accuracy = 0.321. token accuracy = 0.673

dropout = 0.5, RLstrength = 0.001 produces: math accuracy = 0.355. token accuracy = 0.6966666666666667

dropout = 0.55, RLstrength = 0.0012 produces: math accuracy = 0.225, token accuracy = 0.6383333333333333

dropout = 0.55, RLstrength = 0.0012 produces: math accuracy = 0.35, token accuracy = 0.7083333333333334. val_acc = 0.7969765067100525, val_loss = 0.8725050687789917

dropout = 0.55, RLstrength = 0.0004 produces: math accuracy = 0.2, token accuracy = 0.5916666666666667. val_acc = 0.7102698683738708, val_loss = 1.0084364414215088

dropout = 0.55, RLstrength = 0.0002 produces: math accuracy = 0.22, token accuracy = 0.615
val_acc = 0.7328835725784302, val_loss = 1.0302910804748535

